In [4]:
import os
import pandas as pd
from tqdm.auto import tqdm
import openml
from pmlb import fetch_data
from sklearn.model_selection import train_test_split

In [6]:
CONFIG = {
    'min_numeric_prop': 1/3,
    'min_epv': 3,
    'min_instances': 300,
    'drop_ids': [151, 1038, 1467, 1479, 1485, 23517, 40994, 41142, 41144, 41145, 41146, 41158, 41159, 41161, 42769,
                 'PMLB_Hill_Valley_with_noise', 'PMLB_Hill_Valley_without_noise', 'PMLB_ring', 'PMLB_banana', 'PMLB_twonorm', 'PMLB_ionosphere']
}

def compute_dataset_metrics(X, y):
    n_instances, n_features_raw = X.shape
    minority_size = y.value_counts().min()
    
    nunique = X.nunique()
    numeric_cols = X.select_dtypes(include='number').columns
    true_num_count = sum(nunique[c] >= 10 for c in numeric_cols)
    
    ohe_size = sum(max(0, nunique[c] - 1) for c in X.columns if c not in numeric_cols or nunique[c] < 10)
    
    final_dim = true_num_count + ohe_size
    if final_dim == 0:
        return None

    return {
        'total_instances': n_instances,
        'total_features': final_dim,
        'numeric_features': true_num_count,
        'numeric_proportion': round(true_num_count / final_dim, 4),
        'missing_values_proportion': round(X.isna().sum().sum() / (n_instances * n_features_raw), 4) if n_features_raw else 0.0,
        'instances_with_missing_proportion': round(X.isna().any(axis=1).mean(), 4),
        'minority_class_proportion': round(minority_size / n_instances, 4),
        'epv': round((minority_size * 0.9) / final_dim, 4)
    }

def fetch_openml_suite(suite_id, suite_name, cfg):
    suite = openml.study.get_suite(suite_id)
    df_meta = openml.datasets.list_datasets(data_id=suite.data, output_format='dataframe')
    
    mask = (
        (df_meta['NumberOfClasses'] == 2) & 
        (df_meta['NumberOfInstances'] >= cfg['min_instances']) &
        ((df_meta['NumberOfNumericFeatures'] / df_meta['NumberOfFeatures']) >= cfg['min_numeric_prop'])
    )
    candidates = df_meta[mask]
    
    records =[]
    for _, row in tqdm(candidates.iterrows(), total=len(candidates), desc=f"OpenML ({suite_name})"):
        try:
            ds = openml.datasets.get_dataset(row['did'], download_data=True, download_qualities=False, download_features_meta_data=False)
            X, y, _, _ = ds.get_data(dataset_format='dataframe', target=ds.default_target_attribute)
            
            metrics = compute_dataset_metrics(X, y)
            if metrics and metrics['numeric_proportion'] >= cfg['min_numeric_prop'] and metrics['epv'] >= cfg['min_epv']:
                records.append({'dataset_id': row['did'], 'dataset_name': row['name'], 'source_suite': f"OpenML_{suite_name}", **metrics})
        except Exception as e:
            tqdm.write(f"[Error] OpenML '{row['name']}' (ID: {row['did']}) failed: {e}")

    return pd.DataFrame(records)


def fetch_pmlb_datasets(cfg):
    stats_url = "https://raw.githubusercontent.com/EpistasisLab/pmlb/master/pmlb/all_summary_stats.tsv"
    pmlb_meta = pd.read_csv(stats_url, sep='\t')
    
    mask = (
        (pmlb_meta['task'] == 'classification') & 
        (pmlb_meta['n_classes'] == 2) & 
        (pmlb_meta['n_instances'] >= cfg['min_instances']) &
        ((pmlb_meta['n_continuous_features'] / pmlb_meta['n_features']) >= cfg['min_numeric_prop'])
    )
    candidates = pmlb_meta[mask]['dataset'].tolist()
    
    records =[]
    for name in tqdm(candidates, desc="PMLB"):
        try:
            df = fetch_data(name)
            if 'target' not in df.columns or df['target'].nunique() != 2 or len(df) < cfg['min_instances']:
                continue
                
            X, y = df.drop(columns=['target']), df['target']
            
            metrics = compute_dataset_metrics(X, y)
            if metrics and metrics['numeric_proportion'] >= cfg['min_numeric_prop'] and metrics['epv'] >= cfg['min_epv']:
                records.append({'dataset_id': f"PMLB_{name}", 'dataset_name': name, 'source_suite': 'PMLB', **metrics})
        except Exception as e:
            tqdm.write(f"[Error] PMLB '{name}' failed: {e}")

    return pd.DataFrame(records)
    
openml_suites =[
    ('OpenML-CC18', 'CC18'),
    (271, 'AutoML'),
    (14, 'OpenML100')
]

dfs =[fetch_openml_suite(sid, name, CONFIG) for sid, name in openml_suites]
dfs.append(fetch_pmlb_datasets(CONFIG))
    
final_benchmark_df = (
    pd.concat(dfs, ignore_index=True)
    .loc[lambda d: ~d['dataset_id'].isin(CONFIG['drop_ids'])]
    .sort_values(by=['source_suite'], ascending=True)
    .assign(
        temp_name_clean = lambda d: (
            d['dataset_name']
            .str.lower()
            .replace(r'[^a-zA-Z0-9]', '', regex=True)
        )
    )
    .drop_duplicates(subset=['dataset_id'], keep='first')
    .drop_duplicates(subset=['temp_name_clean'], keep='first')
    .drop_duplicates(subset=['total_instances', 'total_features'], keep='first')
    .drop_duplicates(subset=['total_instances', 'minority_class_proportion'], keep='first')
    .drop(columns=['temp_name_clean'])
    .reset_index(drop=True)
)

display(final_benchmark_df)

final_benchmark_df.to_csv('../data/1 - metadata/datasets.csv', index=False)

OpenML (CC18):   0%|          | 0/29 [00:00<?, ?it/s]

OpenML (AutoML):   0%|          | 0/34 [00:00<?, ?it/s]

OpenML (OpenML100):   0%|          | 0/39 [00:00<?, ?it/s]

PMLB:   0%|          | 0/35 [00:00<?, ?it/s]

[Error] PMLB '_deprecated_australian' failed: Dataset not found in PMLB.
[Error] PMLB '_deprecated_breast_cancer_wisconsin' failed: Dataset not found in PMLB.
[Error] PMLB '_deprecated_buggyCrx' failed: Dataset not found in PMLB.
[Error] PMLB '_deprecated_cleve' failed: Dataset not found in PMLB.
[Error] PMLB '_deprecated_credit_a' failed: Dataset not found in PMLB.
[Error] PMLB '_deprecated_crx' failed: Dataset not found in PMLB.
[Error] PMLB '_deprecated_diabetes' failed: Dataset not found in PMLB.
[Error] PMLB '_deprecated_heart_c' failed: Dataset not found in PMLB.
[Error] PMLB '_deprecated_pima' failed: Dataset not found in PMLB.
[Error] PMLB '_deprecated_wdbc' failed: Dataset not found in PMLB.
[Error] PMLB 'breast_cancer_wisconsin_diagnostic' failed: Dataset not found in PMLB.
[Error] PMLB 'credit_approval_australia' failed: Dataset not found in PMLB.
[Error] PMLB 'heart_disease_cleveland' failed: Dataset not found in PMLB.
[Error] PMLB 'horse_colic_surgery' failed: Dataset not 

,dataset_id,dataset_name,source_suite,total_instances,total_features,numeric_features,numeric_proportion,missing_values_proportion,instances_with_missing_proportion,minority_class_proportion,epv
0,1049,pc4,OpenML_AutoML,1458,46,34,0.7391,0.0000,0.0000,0.1221,3.4826
1,41150,MiniBooNE,OpenML_AutoML,130064,50,50,1.0000,0.0000,0.0000,0.2806,656.9820
2,41138,APSFailure,OpenML_AutoML,76000,169,168,0.9941,0.0835,0.9901,0.0181,7.3225
3,40983,wilt,OpenML_AutoML,4839,5,5,1.0000,0.0000,0.0000,0.0539,46.9800
4,40701,churn,OpenML_AutoML,5000,29,16,0.5517,0.0000,0.0000,0.1414,21.9414
5,1494,qsar-biodeg,OpenML_AutoML,1055,71,29,0.4085,0.0000,0.0000,0.3374,4.5127
6,1489,phoneme,OpenML_AutoML,5404,5,5,1.0000,0.0000,0.0000,0.2935,285.4800
7,1486,nomao,OpenML_AutoML,34465,177,74,0.4181,0.0000,0.0000,0.2856,50.0542
8,1464,blood-transfusion-service-center,OpenML_AutoML,748,4,4,1.0000,0.0000,0.0000,0.2380,40.0500
9,1067,kc1,OpenML_AutoML,2109,21,21,1.0000,0.0000,0.0000,0.1546,13.9714


In [7]:
def download_openml_dataset(dataset_id, dataset_name):
    ds = openml.datasets.get_dataset(
        dataset_id, 
        download_data=True, 
        download_qualities=False, 
        download_features_meta_data=False
    )
    X, y, _, _ = ds.get_data(dataset_format='dataframe', target=ds.default_target_attribute)
    
    df = X.copy()
    df['target'] = y
    
    file_path = os.path.join('../data/2 - raw', f"{dataset_name}.csv")
    df.to_csv(file_path, index=False)

def download_pmlb_dataset(dataset_name):
    df = fetch_data(dataset_name)
    
    file_path = os.path.join('../data/2 - raw', f"{dataset_name}.csv")
    df.to_csv(file_path, index=False)


benchmark_path = os.path.join('../data/1 - metadata', 'datasets.csv')
df_benchmark = pd.read_csv(benchmark_path)

for _, row in tqdm(df_benchmark.iterrows(), total=len(df_benchmark), desc="Downloading Datasets"):
    source = str(row['source_suite'])
    d_id = row['dataset_id']
    name = row['dataset_name']
    if source.startswith('OpenML'):
        download_openml_dataset(d_id, name)
        
    elif source == 'PMLB':
        download_pmlb_dataset(name)

In [11]:
import os
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

def standardize_dataframe(df):
    if len(df) > 10000:
        df_processed, _ = train_test_split(
            df, 
            train_size=10000, 
            stratify=df['target'], 
            random_state=42
        )
    else:
        df_processed = df.copy()

    na_proportions = df_processed.isnull().mean()
    cols_to_drop = na_proportions[na_proportions > 0.5].index.tolist()
    
    if 'target' in cols_to_drop:
        cols_to_drop.remove('target')
        
    df_processed = df_processed.drop(columns=cols_to_drop)

    features = [col for col in df_processed.columns if col != 'target']
    num_features = []
    cat_features = []

    for col in features:
        if pd.api.types.is_numeric_dtype(df_processed[col]):
            if df_processed[col].nunique() < 10:
                cat_features.append(col)
            else:
                num_features.append(col)
        else:
            cat_features.append(col)

    rename_map = {}
    num_features.sort()
    cat_features.sort()

    for i, col in enumerate(num_features):
        rename_map[col] = f"NUM_{i:02d}"
    
    for i, col in enumerate(cat_features):
        rename_map[col] = f"CAT_{i:02d}"

    df_standardized = df_processed.rename(columns=rename_map)
    
    cols_ordered = sorted([c for c in df_standardized.columns if c.startswith('NUM')]) + \
                   sorted([c for c in df_standardized.columns if c.startswith('CAT')]) + \
                   ['target']
    
    df_standardized = df_standardized[cols_ordered]

    return df_standardized, rename_map


csv_files = [f for f in os.listdir('../data/2 - raw') if f.endswith('.csv')]

for file in tqdm(csv_files, desc="Processing Datasets"):
    file_path = os.path.join('../data/2 - raw', file)
    
    df_raw = pd.read_csv(file_path)

    df_standardized, rename_map = standardize_dataframe(df_raw)

    schema_info = pd.DataFrame(list(rename_map.items()), columns=['Original_Name', 'Standard_Name'])
    schema_info.to_csv(os.path.join('../data/3 - interim', f"schema_{file}"), index=False)

    output_path = os.path.join('../data/3 - interim', file)
    df_standardized.to_csv(output_path, index=False)

Processing Datasets:   0%|          | 0/32 [00:00<?, ?it/s]